In [0]:

# ============================================================
# 0. CREATING WIDGET TO RECEIVE CONFIG PATH
# ============================================================
dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")





In [0]:
import json

from pyspark.sql import functions as F
from common_utils.logging import get_logger
from common_utils.scdfunctions import scd_type_1

# ============================================================
# 1. LOGGER
# ============================================================
logger = get_logger("b2s_dimcounty")

# ============================================================
# 2. CONFIG PATH
# ============================================================
dbutils.widgets.text(
    "path",
    "/Workspace/Users/shreyassrikanth997@gmail.com/Employee-Data-Platform/src/silver/config/config_dimensions.json"
)
config_path = dbutils.widgets.get("path")

logger.info("Reading Silver configuration from %s", config_path)

# ============================================================
# 3. READ CONFIG
# ============================================================
with open(config_path, "r") as file:
    config = json.load(file)

# ============================================================
# 4. CONFIG SECTIONS
# ============================================================
dimension_config = config["dimcounty"]

source_table = dimension_config["source_table"]
target_table = dimension_config["target_table"]
key_columns  = dimension_config["key_columns"]

# ============================================================
# 5. READ BRONZE TABLE
# ============================================================
logger.info("Reading Bronze table: %s", source_table)

df = spark.table(source_table)

# ============================================================
# 6. TRANSFORMATION FUNCTION
# ============================================================
def transform_dimension(df, dimension_config):

    logger.info("Starting dimension transformation")

    # ---------- DROP COLUMNS ----------
    for column in dimension_config["drop_columns"]:
        df = df.drop(column)

    # ---------- RENAME COLUMNS ----------
    for old_column, new_column in dimension_config["rename_columns"].items():
        df = df.withColumnRenamed(old_column, new_column)

    # ---------- CAST DATE COLUMNS ----------
    for column in dimension_config["date_columns"]:
        df = df.withColumn(column, F.to_date(F.col(column)))

    # ---------- CAST TIMESTAMP COLUMNS ----------
    for column in dimension_config["timestamp_columns"]:
        df = df.withColumn(column, F.to_timestamp(F.col(column)))

    # ---------- STANDARDISE Y / N TO YES / NO ----------
    for column in dimension_config["yes_no_columns"]:
        df = df.withColumn(
            column,
            F.when(F.col(column) == "Y", "Yes")
             .when(F.col(column) == "N", "No")
             .otherwise(F.col(column))
        )

    logger.info("Dimension transformation completed")

    return df

# ============================================================
# 7. TRANSFORM DIMCOUNTY
# ============================================================
dim_county = transform_dimension(df, dimension_config)

# ============================================================
# 8. DISPLAY TRANSFORMED DATA
# ============================================================
display(dim_county)

# ============================================================
# 9. CREATE SILVER SCHEMA IF DOES NOT EXIST
# ============================================================
spark.sql("""
CREATE SCHEMA IF NOT EXISTS employeedataplatform.silver
""")

# ============================================================
# 10. CREATE INITIAL SILVER TABLE IF DOES NOT EXIST
# ============================================================
if not spark.catalog.tableExists(target_table):

    logger.info("Silver table does not exist. Creating initial table")

    dim_county.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    logger.info("Initial Silver table created successfully")

# ============================================================
# 11. PERFORM SCD TYPE 1
# ============================================================
scd_type_1(
    spark,
    dim_county,
    target_table,
    key_columns
)

# ============================================================
# 12. COMPLETED
# ============================================================
logger.info("Silver dimcounty processing completed")